## DLR-PCG Initial X Computational Times

This notebook measures the computational time of the DLR-PCG scheme using different initial guesses $X^{(0)} = U_x \Sigma_x V_x^T$.

The 4 different guesses are
 - `svd`: $X^{(0)} \sim U[0, 10^{-3}]$  (uniform i.i.d), then take the SVD $X^{(0)} = U_x \Sigma_x V_x^T$
 - `qr`: $U_x, V_x = \text{orth}(\hat U_x), \text{orth}(\hat V_x)$, $\hat U_x, \hat V_x \sim U[0, 1]$, $\Sigma_x = \text{diag}(10^{-3} \sigma_i)$ with
    - $\sigma_1 = \tfrac n 2$,
    - $\sigma_i = u \sqrt{n}$, $u \sim U[0, 1]$
    - $X^{(0)} = U_x \Sigma_x V_x^T$  (this mimics the singular values of an uniform iid $X$)
- `low-rank-qr`: Same as `qr`, but truncate $\Sigma_x$ to include only `max_rank` non-zero singular values.
- `householder`: Using Householder transformations to construct $U_x$ and $V_x$, $\Sigma_x$ is as above.

The results show a slight advantage using `qr` or `low-rank-qr` in terms of computational time, but not for the number of iterations. Based on this, I set `X0 = 'qr'` as default.

---

In [1]:
import pandas as pd
from time import time
from utils.problem_setup import TestProblemsSetup
from algorithms.dynamical_low_rank_pcg import DynamicalLowRankPCG

/home/elias/miniforge3/envs/fenics_env/lib/python3.9/site-packages/ufl/__init__.py:250: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
setup = TestProblemsSetup(n=64)
problems = setup.get_test_problems()

In [10]:
p = problems['II']
solver = DynamicalLowRankPCG(p['rsvd'])

repeat = 200
results_jacobi = []
for r in range(repeat):
    for X0 in ['svd', 'qr', 'low-rank-qr', 'householder']:
        t0 = time()
        solver.solve(
            y=p['y'],
            w=p['w'],
            lambda_=1e-8,
            max_rank=2,
            verbose=False,
            X0=X0,
            preconditioner='jacobi'
        )
        t = time() - t0
        results_jacobi.append({'r': r, 'X0': X0, 't': t, 'niter': solver.niter})

results_ic = []
for r in range(repeat):
    for X0 in ['svd', 'qr', 'low-rank-qr', 'householder']:
        t0 = time()
        solver.solve(
            y=p['y'],
            w=p['w'],
            lambda_=1e-8,
            max_rank=2,
            verbose=False,
            X0=X0,
            preconditioner='ic'
        )
        t = time() - t0
        results_ic.append({'r': r, 'X0': X0, 't': t, 'niter': solver.niter})

results_jacobi = pd.DataFrame(results_jacobi)
results_ic = pd.DataFrame(results_ic)

In [11]:
summary_jacobi = results_jacobi.groupby('X0').agg(
    median_t=('t', 'median'),
    mean_niter=('niter', 'mean')
)
summary_ic = results_ic.groupby('X0').agg(
    median_t=('t', 'median'),
    mean_niter=('niter', 'mean')
)

print(summary_jacobi)
print(summary_ic)

             median_t  mean_niter
X0                               
householder  0.205838      205.84
low-rank-qr  0.203577      204.40
qr           0.207603      205.09
svd          0.210344      205.81
             median_t  mean_niter
X0                               
householder  0.071001         3.0
low-rank-qr  0.071470         3.0
qr           0.073400         3.0
svd          0.095970         3.0
